# 06 — Hero Image Bake-Off

**Goal:** Compare cheap image-edit models against the current Gemini baseline for generating studio flat-lay hero images from worn garment photos.

**Pipeline:**
```
1 worn garment photo
  → each model given the same flat-lay prompt
  → side-by-side grid
  → color fidelity + white-pixel ratio metrics
  → cost table
  → decision: cheapest model that matches Gemini quality
```


In [19]:
%matplotlib inline
import os, sys, io, base64
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
from dotenv import load_dotenv

load_dotenv(Path("..") / ".env")
sys.path.insert(0, str(Path("..")))

from lib.providers import GeminiProvider, FalProvider, all_providers, print_availability
from lib.images import resize_image
from lib.metrics import compute_white_pixel_ratio, compute_color_delta_lab

OUT_DIR = Path("../outputs/06")
OUT_DIR.mkdir(parents=True, exist_ok=True)

providers = all_providers()
print("Provider status:")
print_availability(providers)


Provider status:
  gemini       ✓ ready
  openrouter   ✓ ready
  fal          ✓ ready


In [21]:
# ── Load input photo ─────────────────────────────────────────────────────────
import glob

def _load_worn():
    paths = sorted(Path("../inputs/worn").glob("*.jpg")) + sorted(Path("../inputs/worn").glob("*.jpeg")) + sorted(Path("../inputs/worn").glob("*.png"))
    paths = [p for p in paths if not p.name.startswith(".")]
    if paths:
        return paths[0]
    db = sorted(glob.glob(os.path.expanduser("~/.closet/images/items/*.jpg")))
    if db:
        return Path(db[0])
    raise FileNotFoundError("No worn photos in inputs/worn/ and no DB items found")

PHOTO_PATH = _load_worn()
INPUT_IMG  = resize_image(Image.open(PHOTO_PATH))
print(f"Input: {PHOTO_PATH.name}  ({INPUT_IMG.width}×{INPUT_IMG.height})")

plt.figure(figsize=(4, 5))
plt.imshow(INPUT_IMG); plt.title(PHOTO_PATH.name, fontsize=9); plt.axis("off")
plt.tight_layout(); plt.show()


Input: 0VJj34XK0fWoDcNLtv7lD.jpg  (474×493)


[image removed from the public copy: third-party or personal photos]


In [23]:
# ── Hero prompt (same as lib/hero_image.py) ───────────────────────────────────
HERO_PROMPT = """You are generating a clean product photograph of ONE specific garment.

Output a studio flat-lay of THAT SINGLE garment only. No person, no environment, no props.
Neutral light grey or off-white background with a soft natural shadow.

Rules (non-negotiable):
- ONE garment only — the primary subject. Do not add shoes, accessories, or other items.
- SINGLE canonical flat-lay view. No blended angles, no duplicate structural elements.
- Correct anatomy — exactly the sleeves/panels a real garment has. No extras.
- Each logo/print/text appears exactly once in its correct position.
- Preserve color and fabric texture exactly from the reference.

Output one high-resolution flat-lay image."""

print("Prompt:", len(HERO_PROMPT), "chars")


Prompt: 689 chars


In [25]:
# ── fal.ai model arg builders ─────────────────────────────────────────────────
# Each fal model has a different request schema — we normalise here.
from lib.providers import FalProvider
import io, base64

fal = providers["fal"]

def _img_data_url(img: Image.Image) -> str:
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=92)
    return "data:image/jpeg;base64," + base64.b64encode(buf.getvalue()).decode()

def _img_fal_url(img: Image.Image) -> str:
    """Upload to fal storage and return public URL (needed by some models)."""
    return fal.upload_image(img) if fal.available else ""

FAL_MODELS = [
    {
        "id": "fal-ai/qwen-image-edit",
        "args_fn": lambda url: {
            "prompt": HERO_PROMPT,
            "image_url": url,
        },
    },
    {
        "id": "fal-ai/flux-pro/kontext",
        "args_fn": lambda url: {
            "prompt": HERO_PROMPT,
            "image_url": url,
        },
    },
    {
        "id": "fal-ai/bytedance/seedream/v4/edit",
        "args_fn": lambda url: {
            "prompt": HERO_PROMPT,
            "image_url": url,
        },
    },
    {
        "id": "fal-ai/hidream-i1-full",
        "args_fn": lambda url: {
            "prompt": HERO_PROMPT,
            "image_url": url,
        },
    },
]

print("fal.ai models to test:")
for m in FAL_MODELS:
    print(f"  {m['id']}")


fal.ai models to test:
  fal-ai/qwen-image-edit
  fal-ai/flux-pro/kontext
  fal-ai/bytedance/seedream/v4/edit
  fal-ai/hidream-i1-full


In [27]:
# ── Run Gemini baseline ───────────────────────────────────────────────────────
import time

gemini = providers["gemini"]
gemini_results = {}

if gemini.available:
    for model_id in ["gemini-2.5-flash-image", "gemini-3-pro-image-preview"]:
        print(f"[{model_id}]...")
        r = gemini.generate_image(model_id, HERO_PROMPT, images=[INPUT_IMG])
        status = "✓" if r.ok else f"✗ {r.error}"
        print(f"  {status}  {r.latency_s:.1f}s  ${r.cost_usd:.4f}~")
        if r.ok:
            r.image.save(OUT_DIR / f"06_gemini_{model_id.replace('/', '_')}.png")
        gemini_results[model_id] = r
else:
    print("Gemini not available")


[gemini-2.5-flash-image]...
  ✓  9.8s  $0.0390~
[gemini-3-pro-image-preview]...
  ✓  15.3s  $0.0390~


In [28]:
# ── Run fal.ai models ─────────────────────────────────────────────────────────
fal = providers["fal"]
fal_results = {}

if fal.available:
    print("Uploading input image to fal storage...")
    img_url = _img_fal_url(INPUT_IMG)
    print(f"  URL: {img_url[:60]}...")

    for model_spec in FAL_MODELS:
        model_id = model_spec["id"]
        print(f"\n[{model_id}]...")
        args = model_spec["args_fn"](img_url)
        r = fal.generate_image(model_id, args, pricing_kind="image")
        status = "✓" if r.ok else f"✗ {r.error[:80] if r.error else ''}"
        print(f"  {status}  {r.latency_s:.1f}s  ${r.cost_usd:.4f}~")
        if r.ok:
            r.image.save(OUT_DIR / f"06_fal_{model_id.replace('/', '_').replace('-', '_')}.png")
        fal_results[model_id] = r
else:
    print("FAL_KEY not set — skipping fal models")


Uploading input image to fal storage...
  URL: https://v3b.fal.media/files/b/0aa00e33/hQlmjJZCZYaSPZ3SidQR8...

[fal-ai/qwen-image-edit]...
  ✓  8.2s  $0.0350~

[fal-ai/flux-pro/kontext]...
  ✓  10.7s  $0.0400~

[fal-ai/bytedance/seedream/v4/edit]...
  ✗ FalClientHTTPError: [{'type': 'missing', 'loc': ['body', 'image_urls'], 'msg': '  0.4s  $0.0300~

[fal-ai/hidream-i1-full]...
  ✗ timeout after 120s  120.0s  $0.0500~


In [29]:
# ── Side-by-side grid ─────────────────────────────────────────────────────────
all_results = {**gemini_results, **fal_results}
model_ids = list(all_results.keys())
n = len(model_ids)

if n == 0:
    print("No results to display")
else:
    fig, axes = plt.subplots(1, n + 1, figsize=((n + 1) * 4, 6))
    axes[0].imshow(INPUT_IMG)
    axes[0].set_title(f"Input\n{PHOTO_PATH.name[:20]}", fontsize=8)
    axes[0].axis("off")

    for ax, model_id in zip(axes[1:], model_ids):
        r = all_results[model_id]
        if r.ok:
            ax.imshow(r.image)
            ax.set_title(model_id.split("/")[-1][:22] + f"\n${r.cost_usd:.4f} | {r.latency_s:.0f}s", fontsize=7)
        else:
            ax.text(0.1, 0.5, f"FAILED\n{r.error[:60] if r.error else ''}", 
                    transform=ax.transAxes, fontsize=7, va="center")
            ax.set_title(model_id.split("/")[-1][:22], fontsize=7)
        ax.axis("off")

    plt.suptitle("Hero image bake-off — all models", fontsize=11)
    plt.tight_layout(); plt.show()


[image removed from the public copy: third-party or personal photos]


In [30]:
# ── Metrics table ─────────────────────────────────────────────────────────────
rows = []
all_results_combined = {**gemini_results, **fal_results}
for model_id, r in all_results_combined.items():
    white = compute_white_pixel_ratio(r.image) if r.ok else None
    delta = compute_color_delta_lab(INPUT_IMG, r.image) if r.ok else None
    rows.append({
        "Model": model_id.split("/")[-1][:28],
        "Provider": r.provider,
        "Latency (s)": f"{r.latency_s:.1f}",
        "Cost": f"${r.cost_usd:.4f}{'~' if not r.cost_is_actual else ''}",
        "BG white ratio": f"{white:.3f}" if white is not None else "—",
        "Color Δ LAB": f"{delta:.1f}" if delta is not None else "—",
        "Eyeball (1-5)": "—",
        "Status": "✓" if r.ok else "✗",
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))
df.to_csv(OUT_DIR / "06_metrics.csv", index=False)
print("\n~ = static estimate  (fill Eyeball column manually after inspecting outputs)")


                     Model Provider Latency (s)     Cost BG white ratio Color Δ LAB Eyeball (1-5) Status
    gemini-2.5-flash-image   gemini         9.8 $0.0390~          0.000        18.2             —      ✓
gemini-3-pro-image-preview   gemini        15.3 $0.0390~          0.000        16.3             —      ✓
           qwen-image-edit      fal         8.2 $0.0350~          0.001        15.1             —      ✓
                   kontext      fal        10.7 $0.0400~          0.000        15.7             —      ✓
                      edit      fal         0.4 $0.0300~              —           —             —      ✗
           hidream-i1-full      fal       120.0 $0.0500~              —           —             —      ✗

~ = static estimate  (fill Eyeball column manually after inspecting outputs)


## Decision note

Fill in after reviewing outputs and eyeball scores.

**Winner:** `_______`

**Why:**
- Color fidelity (Δ LAB vs Gemini baseline): 
- BG cleanliness: 
- Eyeball score: 
- Cost per image vs Gemini: 

**Ship recommendation for `src/ai/heroImage.ts`:**
```
provider: fal  (or gemini if no fal model matches quality)
model:    _______
```

**Open questions:**
- Does the winner handle dark/textured garments as well as light ones?
- Does multi-image input help any fal model the way it helps Gemini?
